In [ ]:
pip install ultralytics


Note: you may need to restart the kernel to use updated packages.


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="Yk7jEOUD50JWpOexZ6qT")
project = rf.workspace("objectdetectiondsp").project("nameplate-car-tb60i")
version = project.version(1)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...


In [14]:
from ultralytics import YOLO

# Load model
model = YOLO("yolov8l.pt")  # or yolov8n.pt, yolov8s.pt etc.

# Train
model.train(
    data="C:/Users/Dudut/dataset/data.yaml",
    epochs=10,
    imgsz=640,
    batch=6,
    plots=True
)


RuntimeError: PytorchStreamReader failed reading zip archive: failed finding central directory

In [5]:
import cv2
from ultralytics import YOLO
#import numpy as np
import matplotlib.pyplot as plt

In [6]:
!pip install pytesseract

In [9]:
from ultralytics import YOLO
import cv2
import pytesseract
import matplotlib.pyplot as plt
import os

# Optional: Set path to Tesseract executable if needed
# pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Configuration
MODEL_PATH = "runs/detect/train3/weights/best.pt"
IMAGE_PATH = "plate11.jpg"
CONFIDENCE_THRESHOLD = 0.001
IMAGE_SIZE = 1280
IOU_THRESHOLD = 0.3
SHOW_DEBUG = True  # Set to False to disable visualization

# Load YOLO model
model = YOLO(MODEL_PATH)

def preprocess_for_ocr(roi):
    """Preprocess cropped license plate image for better OCR accuracy."""
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

    # Apply CLAHE to enhance contrast
    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
    gray = clahe.apply(gray)

    # Denoise and threshold
    gray = cv2.medianBlur(gray, 3)
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return thresh

def perform_ocr(image):
    """Run Tesseract OCR on preprocessed image."""
    config = r"--oem 3 --psm 6 -c tessedit_char_whitelist=ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
    text = pytesseract.image_to_string(image, config=config)
    return text.strip()

def detect_and_read_plate(image_path):
    """Detect license plates using YOLO and read them using OCR."""
    results = model.predict(image_path, conf=CONFIDENCE_THRESHOLD, imgsz=IMAGE_SIZE, iou=IOU_THRESHOLD)

    # Read and prepare image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plate_number = ""

    for result in results:
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            roi = img[y1:y2, x1:x2]

            # Preprocess and OCR
            ocr_input = preprocess_for_ocr(roi)
            plate_number = perform_ocr(ocr_input)

            # Draw pink box and label
            cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (255, 0, 255), 3)  # Pink
            cv2.putText(img_rgb, plate_number, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 0, 255), 2)  # Pink

            # Optional debug visualization
            if SHOW_DEBUG:
                plt.figure(figsize=(12, 4))
                plt.subplot(1, 3, 1)
                plt.imshow(img_rgb)
                plt.title("YOLO + OCR")
                plt.axis("off")

                plt.subplot(1, 3, 2)
                plt.imshow(cv2.cvtColor(roi, cv2.COLOR_BGR2RGB))
                plt.title("Cropped Plate")
                plt.axis("off")

                plt.subplot(1, 3, 3)
                plt.imshow(ocr_input, cmap="gray")
                plt.title("OCR Input (Enhanced)")
                plt.axis("off")
                plt.show()

            return plate_number  # Only process the first detection

    return None

# ---- MAIN EXECUTION ----
if __name__ == "__main__":
    plate = detect_and_read_plate(IMAGE_PATH)
    if plate:
        print("Extracted Plate Number:", plate)
    else:
        print("No plate detected.")

FileNotFoundError: [Errno 2] No such file or directory: 'runs\\detect\\train3\\weights\\best.pt'